# Cheatsheet Praktikum — Klasifikasi Teks (TF-IDF + Classical ML)

**IF5153 Advanced NLP** · referensi slide `02a-Klasifikasi-Teks` & `02b-Word-Representation` (Ayu Purwarianti)

Semua contoh membaca **file nyata di folder `data/`**, bukan data yang ditempel di dalam kode —
supaya alurnya sama persis dengan praktikum: dapat file → baca → bersihkan → latih.
Tinggal Run All, lalu copy-paste blok yang dibutuhkan.

```
lat-prak1/
├── cheatsheet_klasifikasi_teks.ipynb
└── data/
    ├── sms_spam.csv              80 SMS EN, header, koma, biner (spam/ham)
    ├── ulasan_produk_id.csv      40 ulasan ID, header, koma, biner (positif/negatif)
    ├── berita_topik.tsv          88 judul berita, TAB, 4 kelas (multi-class)
    ├── sentiment140_sample.csv   3.200 tweet, TANPA header, latin-1, label 0/4
    ├── reviews_messy.csv         18 baris kotor: NaN, duplikat, label tak konsisten
    └── split/                    kasus "train & test sudah dipisah dosen" -> §2b
        ├── train.csv             2.400 baris berlabel
        ├── test.csv              800 baris berlabel
        └── test_unlabeled.csv    800 baris TANPA label (gaya kompetisi)
```

`sentiment140_sample.csv` adalah sampel acak berimbang dari dataset Sentiment140 yang kamu pakai
di HW-2 (aslinya 1,6 juta baris) — dipakai supaya ada satu contoh data **nyata dan berisik**,
bukan cuma data mainan yang skornya selalu 1.00.

**Kebutuhan:** `scikit-learn`, `pandas`, `numpy`, `nltk` (opsional — ada fallback kalau NLTK/koneksi tidak ada).

```bash
pip install scikit-learn pandas numpy nltk
```

### Cara menjalankan

Notebook harus dijalankan **dari folder `lat-prak1`** (path data ditulis relatif, `data/...`).

```bash
jupyter lab cheatsheet_klasifikasi_teks.ipynb
```

Atau buka langsung di VS Code, lalu **Run All**. Untuk memastikan semuanya jalan tanpa membuka UI:

```bash
python -m nbconvert --to notebook --execute --inplace cheatsheet_klasifikasi_teks.ipynb
```

Perintah itu menjalankan seluruh notebook dan gagal dengan pesan error kalau ada satu sel pun
yang bermasalah. Seluruh notebook ini sudah lolos perintah tersebut: 39 sel kode, 0 error.
Waktu jalan total sekitar satu menit (paling lama §8 `GridSearchCV`).
Efek samping yang normal: §2b menulis file `submission.csv` di folder ini.

### Daftar isi

| § | Isi | Status |
|---|---|---|
| 0 | Blok kilat: pipeline lengkap dalam 10 baris | — |
| 1 | Peta materi: pipeline klasifikasi teks (teori) | — |
| 2 | Data: loader serba guna untuk segala bentuk input + pembersihan | **wajib** |
| 2b | **Kalau train & test sudah dipisah dosen (dua file)** | situasional |
| 3 | Preprocessing: 3a EN · 3b ID · 3c sambung ke vectorizer · 3d sisa langkah slide · **3e alur yang dipakai** | opsional |
| 4 | Feature extraction: BoW, TF, TF-IDF, n-gram | **wajib** |
| 5 | Reduksi & seleksi fitur (df, chi2, mutual information) | opsional |
| 6 | Classifier: NB, Decision Tree, LogReg, SVM · 6b multi-class · 6c data nyata | **wajib** |
| 7 | Evaluasi: precision / recall / F1 / confusion matrix | **wajib** |
| 8 | Tuning: GridSearchCV | opsional |
| 9 | Interpretasi model: fitur paling menentukan | opsional |
| 10 | Error yang sering muncul & solusinya | — |
| 11 | Lampiran: rumus + LSA/SVD | opsional |
| 12 | Checklist saat praktikum | — |

### Wajib vs opsional

Hanya **empat** hal yang benar-benar wajib: baca data → ubah teks jadi angka → latih model →
ukur. Semua sisanya penyempurnaan. Pipeline paling minim yang sudah sah dilaporkan:

```python
df = pd.read_csv("data.csv")                                   # 1. baca         (WAJIB)
X_tr, X_te, y_tr, y_te = train_test_split(df.text, df.label,   # 2. pisah        (WAJIB)
                                          stratify=df.label, random_state=42)
m = Pipeline([("tfidf", TfidfVectorizer()),                    # 3. vectorize    (WAJIB)
              ("clf", MultinomialNB())]).fit(X_tr, y_tr)       # 4. latih        (WAJIB)
print(classification_report(y_te, m.predict(X_te)))            # 5. ukur         (WAJIB)
```

| Langkah | Status | Kalau dilewati, apa yang terjadi |
|---|---|---|
| Baca & bersihkan data (§2) | **WAJIB** | tanpa data tidak ada apa-apa; tanpa dibersihkan skornya palsu |
| Pisah train/test (§2 atau §2b) | **WAJIB** | tidak bisa mengukur generalisasi sama sekali |
| Preprocessing (§3) | opsional | **tetap jalan** — `TfidfVectorizer` sudah lowercase + tokenisasi sendiri |
| Vectorization (§4) | **WAJIB** | model sklearn tidak menerima string mentah → `ValueError` |
| Seleksi fitur (§5) | opsional | tetap jalan, hanya lebih lambat & sedikit lebih rentan overfit |
| Latih classifier (§6) | **WAJIB** | — |
| Evaluasi (§7) | **WAJIB** | tidak ada yang bisa dilaporkan |
| Tuning (§8) | opsional | parameter default sklearn biasanya sudah lumayan |
| Interpretasi (§9) | opsional | tetap jalan; berguna untuk bagian analisis di laporan |
| LSA/SVD (§11) | opsional | murni tambahan |

Aturan praktisnya: **jalankan pipeline minimum dulu sampai keluar angka**, baru tambahkan langkah
opsional satu per satu sambil mencatat perubahan skornya. Jangan menumpuk semua preprocessing di
awal — kalau skornya jelek kamu tidak akan tahu penyebabnya yang mana.

---
## §0 · Blok kilat — kalau mepet, ini saja

Satu sel self-contained. Ganti `texts` / `labels` dengan dataset praktikum, selesai.

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

texts = ["free prize claim now", "win free cash claim urgent", "you won a free prize holiday",
         "claim your free discount now urgent", "urgent win cash prize now",
         "free holiday claim your prize",
         "see you at the meeting tomorrow", "thanks for the notes see you",
         "the lecture is tomorrow at nine", "see you tomorrow at the library",
         "thanks for the meeting notes", "the assignment deadline is tomorrow"]
labels = ["spam"] * 6 + ["ham"] * 6

X_tr, X_te, y_tr, y_te = train_test_split(
    texts, labels, test_size=1/3, random_state=42, stratify=labels)   # stratify: proporsi kelas terjaga

model = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1)),
    ("clf",   MultinomialNB()),
])
model.fit(X_tr, y_tr)
print(classification_report(y_te, model.predict(X_te), zero_division=0))
print(model.predict(["urgent free cash claim", "see you at the lecture"]))

              precision    recall  f1-score   support

         ham       1.00      1.00      1.00         2
        spam       1.00      1.00      1.00         2

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4

['spam' 'ham']


> **Kenapa `Pipeline`?** Vectorizer di-`fit` **hanya** pada data latih, lalu otomatis dipakai
> `transform` saja pada data uji. Kalau vectorizer di-`fit` pada seluruh data, vocabulary dan
> nilai IDF ikut "melihat" data uji → **data leakage**, skor jadi terlalu bagus dan salah.

---
## §1 · Peta materi (teori, slide 02a)

### Pipeline inti

```
Input teks  →  PREPROCESSING  →  FEATURE EXTRACTION  →  CLASSIFICATION  →  label
              (teks → token)     (token → angka)       (rule / model)
```

Slide "Machine Learning for NLP" membagi ini jadi dua jalur yang **harus simetris**:

```
Training data → Preprocess → Feature Extraction → Model Training → NLP Model
                                     ↑
                          Word Representation Model
                                     ↓
Testing data  → Preprocess → Feature Extraction → Model Inference → label
```

Kunci: langkah preprocessing dan vectorizer di jalur inference **wajib persis sama** dengan
jalur training. Di sklearn hal ini dijamin oleh `Pipeline`.

### Karakterisasi task (slide 2)
| Dimensi | Pilihan |
|---|---|
| Output | single label vs **multi label** |
| Panjang input | pesan pendek (SMS/tweet) vs dokumen |
| Ragam bahasa | *user generated content* (informal, typo, slang) vs kalimat formal |
| Kelengkapan | kalimat utuh vs potongan (mis. respons chat) |

### Dua gaya klasifikasi (slide 6)
| | Rule based | Statistical / ML |
|---|---|---|
| Aturan | ditulis manusia (mis. *spam word list* + threshold) | dipelajari otomatis dari **training data** |
| Kelebihan | transparan, tanpa data berlabel | menangkap pola kompleks, skalabel |
| Kekurangan | rapuh, mahal dirawat | butuh data berlabel, kurang transparan |

### Kenapa spam word list gagal (slide 5) — sering jadi soal
1. Kata yang muncul di spam sering **kata umum** ("this", "and") → perlu **stopword elimination**.
2. Kata spam juga muncul di ham ("hotel", "price") → perlu **pembobotan / seleksi fitur** (TF-IDF, MI).
3. **Urutan kata** mengubah label → unigram tidak cukup, perlu **n-gram**.
   - "Click here now to claim your free $500 gift card before it expires" → SPAM
   - "Thanks for shopping with us, here's a $500 gift card, no action needed" → NOT SPAM

---
## §2 · Data — memuat input apa pun  ·  `WAJIB`

Notebook ini **tidak menaruh data di dalam kode**. Semua contoh membaca file di folder `data/`,
persis seperti nanti di praktikum. Isi folder sengaja dibuat berbeda-beda bentuknya supaya
tiap kemungkinan input ada contohnya:

| File | Bentuk | Kenapa ada |
|---|---|---|
| `data/sms_spam.csv` | header `label,message`, koma, UTF-8, biner (spam/ham) | kasus paling umum & paling rapi |
| `data/ulasan_produk_id.csv` | header `text,label`, koma, UTF-8, bahasa **Indonesia** | urutan kolom terbalik + bahasa lain |
| `data/berita_topik.tsv` | header `kategori\tjudul`, **tab**, 4 kelas | **multi-class**, separator bukan koma, kolom label di depan |
| `data/sentiment140_sample.csv` | **tanpa header**, 6 kolom, **latin-1**, label angka `0`/`4` | data Kaggle mentah (sampel 3.200 baris dari dataset HW-2) |
| `data/reviews_messy.csv` | ada NaN, duplikat, label tak konsisten, kolom tak terpakai | latihan pembersihan data |

Kalau praktikum memberi file lain, taruh saja di `data/` lalu panggil `load_text_dataset()` di bawah.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("data")
if not DATA_DIR.exists():                       # jaga-jaga kalau notebook dijalankan dari folder lain
    DATA_DIR = Path(__file__).parent / "data" if "__file__" in dir() else Path("../data")

for p in sorted(DATA_DIR.glob("*")):
    print(f"{p.name:32s} {p.stat().st_size/1024:8.1f} KB")

berita_multilabel.csv                 6.9 KB
berita_topik.tsv                      6.7 KB
keluhan_imbalanced.csv              149.0 KB
reviews_messy.csv                     0.9 KB
sentiment140_sample.csv             437.0 KB
sms_spam.csv                          6.8 KB
split                                 0.0 KB
ulasan_produk_id.csv                  3.4 KB


### Loader serba guna

Fungsi di bawah menangani hal-hal yang biasanya bikin macet di 10 menit pertama praktikum:
separator, encoding, ada/tidaknya header, nama kolom yang tidak diketahui, label berupa angka,
baris kosong, dan data yang terlalu besar.

In [3]:
def load_text_dataset(path, text_col=None, label_col=None, sep=None, header="infer",
                      encoding=None, label_map=None, min_len=3, sample=None,
                      random_state=42, verbose=True):
    # Baca dataset teks apa pun -> DataFrame dengan tepat 2 kolom: 'text' dan 'label'.
    #   text_col / label_col : nama atau indeks kolom. None = dideteksi otomatis.
    #   sep                  : None = tebak dari ekstensi (.tsv/.tab -> tab, sisanya koma).
    #   header               : "infer" (ada header) atau None (tanpa header).
    #   encoding             : None = coba utf-8, lalu latin-1, lalu cp1252.
    #   label_map            : dict pemetaan label, mis. {0: "negatif", 4: "positif"}.
    #   min_len              : buang teks lebih pendek dari sekian karakter.
    #   sample               : ambil n baris saja (stratified) supaya cepat saat eksplorasi.
    path = Path(path)
    if sep is None:
        sep = "\t" if path.suffix.lower() in {".tsv", ".tab"} else ","

    encodings = [encoding] if encoding else ["utf-8", "latin-1", "cp1252"]
    df = last_err = None
    for enc in encodings:
        try:
            df = pd.read_csv(path, sep=sep, header=header, encoding=enc,
                             engine="python", on_bad_lines="skip")
            if verbose:
                print(f"[baca] {path.name} sep={sep!r} encoding={enc} -> {df.shape}")
            break
        except (UnicodeDecodeError, UnicodeError) as e:
            last_err = e
    if df is None:
        raise last_err

    # --- tentukan kolom teks: kolom berisi string dengan rata-rata panjang terbesar
    #     (cek isinya, bukan dtype -- pandas 3 memakai dtype 'str', pandas 2 memakai 'object')
    if text_col is None:
        teks_cols = [c for c in df.columns
                     if df[c].map(lambda v: isinstance(v, str)).mean() > 0.5]
        if not teks_cols:
            raise ValueError("Tidak ada kolom teks. Sebutkan text_col secara eksplisit.")
        text_col = max(teks_cols, key=lambda c: df[c].astype(str).str.len().mean())
    # --- tentukan kolom label: kolom lain dengan jumlah nilai unik paling sedikit (2..20)
    if label_col is None:
        kandidat = [(df[c].nunique(dropna=True), c) for c in df.columns
                    if c != text_col and 2 <= df[c].nunique(dropna=True) <= 20]
        if not kandidat:
            raise ValueError("Kolom label tidak terdeteksi. Sebutkan label_col secara eksplisit.")
        label_col = min(kandidat)[1]
    if verbose:
        print(f"[kolom] text={text_col!r}  label={label_col!r}")

    out = df[[text_col, label_col]].copy()
    out.columns = ["text", "label"]

    # --- bersihkan
    n0 = len(out)
    out["text"] = out["text"].astype(str).str.strip()
    out["label"] = out["label"].apply(lambda v: v.strip().lower() if isinstance(v, str) else v)
    if label_map:
        out["label"] = out["label"].map(label_map).fillna(out["label"])
    out = out[out["text"].str.len() >= min_len]
    out = out[~out["text"].str.lower().isin({"nan", "none", "na", "-"})]
    out = out.dropna(subset=["text", "label"])
    out = out.drop_duplicates(subset=["text"])
    out = out.reset_index(drop=True)

    if sample and sample < len(out):
        from sklearn.model_selection import train_test_split
        out, _ = train_test_split(out, train_size=sample, stratify=out["label"],
                                  random_state=random_state)
        out = out.reset_index(drop=True)

    if verbose:
        print(f"[bersih] {n0} -> {len(out)} baris | distribusi: {out['label'].value_counts().to_dict()}")
    return out

### Contoh 1 — CSV rapi berheader (kasus paling umum)

In [4]:
df_en = load_text_dataset(DATA_DIR / "sms_spam.csv")
df_en.head(3)

[baca] sms_spam.csv sep=',' encoding=utf-8 -> (80, 2)
[kolom] text='message'  label='label'
[bersih] 80 -> 80 baris | distribusi: {'ham': 40, 'spam': 40}


,text,label
0,"My train is delayed again, I should arrive by ...",ham
1,"Thanks for helping with the presentation, it w...",ham
2,Special discount tickets for the theatre this ...,spam


### Contoh 2 — bahasa Indonesia, urutan kolom terbalik

In [5]:
df_id = load_text_dataset(DATA_DIR / "ulasan_produk_id.csv")
df_id.sample(3, random_state=0)

[baca] ulasan_produk_id.csv sep=',' encoding=utf-8 -> (40, 2)
[kolom] text='text'  label='label'
[bersih] 40 -> 40 baris | distribusi: {'negatif': 20, 'positif': 20}


,text,label
22,"Rasanya enak dan porsinya besar, tempatnya jug...",positif
20,"Baru sebulan sudah harus servis, kualitas kont...",negatif
25,"Fiturnya lengkap, tampilannya simpel, tidak bi...",positif


### Contoh 3 — TSV multi-class (4 kelas, separator tab)

In [6]:
df_multi = load_text_dataset(DATA_DIR / "berita_topik.tsv")
print(df_multi["label"].value_counts().to_dict())
df_multi.head(3)

[baca] berita_topik.tsv sep='\t' encoding=utf-8 -> (88, 2)
[kolom] text='judul'  label='kategori'
[bersih] 88 -> 88 baris | distribusi: {'teknologi': 22, 'politik': 22, 'ekonomi': 22, 'olahraga': 22}
{'teknologi': 22, 'politik': 22, 'ekonomi': 22, 'olahraga': 22}


,text,label
0,Ponsel lipat generasi kedua hadir dengan engse...,teknologi
1,Layanan komputasi awan perusahaan itu mengalam...,teknologi
2,Mobil listrik otonom mulai diuji coba di jalan...,teknologi


### Contoh 4 — CSV Kaggle mentah: tanpa header, latin-1, label angka

Ini format Sentiment140 (dataset HW-2): 6 kolom `target,id,date,flag,user,text`, tanpa baris
header, encoding `latin-1`, dan label `0` = negatif, `4` = positif. Dua cara: biarkan
kolomnya dideteksi otomatis, atau sebutkan indeks kolomnya (lebih aman kalau formatnya sudah tahu).

In [7]:
# Cara otomatis -- kolom 5 (teks terpanjang) & kolom 0 (unik paling sedikit)
df_s140 = load_text_dataset(DATA_DIR / "sentiment140_sample.csv", header=None,
                            label_map={0: "negatif", 4: "positif"})
print()

# Cara eksplisit -- disarankan kalau format sudah diketahui
df_s140 = load_text_dataset(DATA_DIR / "sentiment140_sample.csv", header=None,
                            text_col=5, label_col=0,
                            label_map={0: "negatif", 4: "positif"}, sample=2000)
df_s140.head(3)

[baca] sentiment140_sample.csv sep=',' encoding=latin-1 -> (3200, 6)
[kolom] text=5  label=0
[bersih] 3200 -> 3200 baris | distribusi: {'negatif': 1600, 'positif': 1600}

[baca] sentiment140_sample.csv sep=',' encoding=latin-1 -> (3200, 6)
[kolom] text=5  label=0
[bersih] 3200 -> 2000 baris | distribusi: {'negatif': 1000, 'positif': 1000}


,text,label
0,"You'e going,so at least I might be able to get...",negatif
1,watched grey's and svu after a nice night of b...,positif
2,Lolz they have rearranged the seat reservation...,positif


### Contoh 5 — data kotor: NaN, duplikat, label tak konsisten

Yang dibersihkan dan **kenapa**:

| Masalah | Akibat kalau dibiarkan | Penanganan |
|---|---|---|
| Teks kosong / `"NA"` | vectorizer menghasilkan baris nol, model bingung | filter `min_len`, buang literal `nan`/`na` |
| Duplikat | dokumen yang sama bisa masuk train **dan** test → skor palsu (leakage) | `drop_duplicates` **sebelum** split |
| Label `"Positif"` / `" NEGATIF "` / `"negatif "` | dianggap kelas berbeda-beda | `.str.strip().str.lower()` |
| Kolom `id`, `tanggal` | bukan fitur teks, ikut terbawa kalau asal `pd.read_csv` | ambil hanya 2 kolom yang dipakai |

In [8]:
mentah = pd.read_csv(DATA_DIR / "reviews_messy.csv")
print("SEBELUM:", mentah.shape)
print(mentah["sentimen"].value_counts(dropna=False).to_dict())

df_bersih = load_text_dataset(DATA_DIR / "reviews_messy.csv",
                              text_col="review", label_col="sentimen", min_len=5)
print("\nSESUDAH:", df_bersih.shape)
df_bersih

SEBELUM: (18, 4)
{'positif': 6, 'Positif': 3, 'negatif': 2, 'Negatif': 2, ' NEGATIF ': 1, 'POSITIF': 1, 'negatif ': 1, nan: 1, 'NEGATIF': 1}
[baca] reviews_messy.csv sep=',' encoding=utf-8 -> (18, 4)
[kolom] text='review'  label='sentimen'
[bersih] 18 -> 13 baris | distribusi: {'positif': 7, 'negatif': 6}

SESUDAH: (13, 2)


,text,label
0,"Barangnya bagus banget, pengiriman cepat",positif
1,"barangnya bagus banget, pengiriman cepat",positif
2,"Kecewa, barang rusak pas sampai",negatif
3,Pelayanan lambat dan tidak ramah sama sekali,negatif
4,"Produknya sesuai deskripsi, saya puas",positif
5,Harga mahal kualitas biasa saja,negatif
6,Mantap!!! 👍👍👍,positif
7,"Tidak sesuai gambar, mengecewakan sekali",negatif
8,"Pengiriman cepat, seller ramah, recommended",positif
9,"Barang tidak pernah sampai, uang tidak kembali",negatif


### Kalau inputnya bukan CSV

```python
# Excel
df = pd.read_excel("data.xlsx", sheet_name=0)

# JSON / JSON Lines
df = pd.read_json("data.json")
df = pd.read_json("data.jsonl", lines=True)

# Satu folder per kelas, isinya file .txt  ->  data/pos/*.txt, data/neg/*.txt
from sklearn.datasets import load_files
bunch = load_files("data", encoding="utf-8", decode_error="replace")
df = pd.DataFrame({"text": bunch.data, "label": [bunch.target_names[i] for i in bunch.target]})

# Dataset bawaan sklearn (tanpa unduh manual, enak buat latihan cepat)
from sklearn.datasets import fetch_20newsgroups
tr = fetch_20newsgroups(subset="train", categories=["sci.space", "rec.autos"])
df = pd.DataFrame({"text": tr.data, "label": [tr.target_names[i] for i in tr.target]})

# HuggingFace datasets (kalau diminta pakai dataset Indonesia, mis. IndoNLU SmSA)
# from datasets import load_dataset
# ds = load_dataset("indonlp/indonlu", "smsa")
# df = pd.DataFrame(ds["train"])

# File teks polos, satu baris satu dokumen, label ada di file terpisah
texts  = Path("docs.txt").read_text(encoding="utf-8").splitlines()
labels = Path("labels.txt").read_text(encoding="utf-8").splitlines()
df = pd.DataFrame({"text": texts, "label": labels})
```

### Cek wajib setelah data masuk

In [9]:
def cek_dataset(df, nama="dataset"):
    print(f"=== {nama} ===")
    print("jumlah baris :", len(df))
    print("distribusi   :", df["label"].value_counts().to_dict())
    print("imbalance    : rasio kelas terbesar/terkecil =",
          round(df["label"].value_counts().max() / df["label"].value_counts().min(), 2))
    print("panjang teks : rata-rata", int(df["text"].str.split().str.len().mean()),
          "kata | maks", int(df["text"].str.split().str.len().max()))
    print("teks kosong  :", int((df["text"].str.strip() == "").sum()),
          "| duplikat:", int(df["text"].duplicated().sum()))
    print("contoh       :", df["text"].iloc[0][:90], "...")
    print()

for nama, d in [("sms_spam (EN)", df_en), ("ulasan (ID)", df_id),
                ("berita (multi-class)", df_multi), ("sentiment140", df_s140)]:
    cek_dataset(d, nama)

=== sms_spam (EN) ===
jumlah baris : 80
distribusi   : {'ham': 40, 'spam': 40}
imbalance    : rasio kelas terbesar/terkecil = 1.0
panjang teks : rata-rata 13 kata | maks 20
teks kosong  : 0 | duplikat: 0
contoh       : My train is delayed again, I should arrive by six if nothing else happens ...

=== ulasan (ID) ===
jumlah baris : 40
distribusi   : {'negatif': 20, 'positif': 20}
imbalance    : rasio kelas terbesar/terkecil = 1.0
panjang teks : rata-rata 10 kata | maks 15
teks kosong  : 0 | duplikat: 0
contoh       : Bahannya tipis dan jahitannya berantakan, tidak sebanding dengan harganya ...

=== berita (multi-class) ===
jumlah baris : 88
distribusi   : {'teknologi': 22, 'politik': 22, 'ekonomi': 22, 'olahraga': 22}
imbalance    : rasio kelas terbesar/terkecil = 1.0
panjang teks : rata-rata 9 kata | maks 11
teks kosong  : 0 | duplikat: 0
contoh       : Ponsel lipat generasi kedua hadir dengan engsel lebih tahan lama ...

=== sentiment140 ===
jumlah baris : 2000
distribusi   : {'negati

---
## §2b · Kalau train & test SUDAH dipisah dosen  ·  `WAJIB kalau formatnya begini`

Kasus yang sangat umum: dosen membagikan **dua file**, `train.csv` dan `test.csv`. Contohnya ada
di `data/split/`. Yang berubah dari alur biasa hanya satu: **jangan panggil `train_test_split`
lagi**. Sisanya sama.

| Beda | Satu file | Dua file (train + test) |
|---|---|---|
| Pemisahan | `train_test_split(...)` sendiri | **sudah dipisah**, jangan dipisah lagi |
| Vectorizer | `fit` pada bagian train hasil split | `fit` pada **seluruh isi train.csv** |
| Data uji | hasil split | seluruh isi `test.csv`, hanya di-`transform` |
| Validasi | test set | **buat validation set dari train**, atau cross-validation di train |

**Aturan yang tidak boleh dilanggar:** `test.csv` disentuh **hanya sekali di akhir**, untuk
melaporkan skor. Semua eksperimen (pilih preprocessing, tuning, bandingkan model) dilakukan
pada train saja lewat validation split atau cross-validation. Kalau kamu bolak-balik menyetel
parameter sambil melihat skor test, kamu sedang overfit ke test set.

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import f1_score

SPLIT_DIR = DATA_DIR / "split"
print([p.name for p in sorted(SPLIT_DIR.glob("*"))])

LABELS = {0: "negatif", 4: "positif"}
train = load_text_dataset(SPLIT_DIR / "train.csv", label_map=LABELS)
test  = load_text_dataset(SPLIT_DIR / "test.csv",  label_map=LABELS)

print("\ntrain:", train.shape, "| test:", test.shape)
print("distribusi train:", train["label"].value_counts().to_dict())
print("distribusi test :", test["label"].value_counts().to_dict())

['test.csv', 'test_unlabeled.csv', 'train.csv']


[baca] train.csv sep=',' encoding=utf-8 -> (2400, 2)


[kolom] text='text'  label='target'
[bersih] 2400 -> 2400 baris | distribusi: {'positif': 1200, 'negatif': 1200}
[baca] test.csv sep=',' encoding=utf-8 -> (800, 2)
[kolom] text='text'  label='target'
[bersih] 800 -> 800 baris | distribusi: {'positif': 400, 'negatif': 400}

train: (2400, 2) | test: (800, 2)
distribusi train: {'positif': 1200, 'negatif': 1200}
distribusi test : {'positif': 400, 'negatif': 400}


### Cek kesehatan split (lakukan sebelum melatih apa pun)

Tiga hal yang bisa membuat laporanmu salah total kalau tidak diperiksa.

In [11]:
# 1) Kebocoran: dokumen yang sama muncul di train DAN test
irisan = set(train["text"]) & set(test["text"])
print("dokumen bocor train<->test :", len(irisan))
if irisan:
    test = test[~test["text"].isin(irisan)]     # buang dari TEST, bukan dari train
    print("  -> dibuang dari test, sisa:", len(test))

# 2) Label yang muncul di test tapi tak pernah ada di train (model mustahil menebaknya)
print("label train:", sorted(train['label'].unique()))
print("label test :", sorted(test['label'].unique()))
print("label asing di test:", set(test["label"]) - set(train["label"]))

# 3) Distribusi kelas mirip atau tidak
print("proporsi train:", train["label"].value_counts(normalize=True).round(3).to_dict())
print("proporsi test :", test["label"].value_counts(normalize=True).round(3).to_dict())

dokumen bocor train<->test : 0
label train: ['negatif', 'positif']
label test : ['negatif', 'positif']
label asing di test: set()
proporsi train: {'positif': 0.5, 'negatif': 0.5}
proporsi test : {'positif': 0.5, 'negatif': 0.5}


### Latih di train, ukur sekali di test

In [12]:
pipe_split = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("clf",   LogisticRegression(max_iter=1000)),
])

# fit HANYA pada train -- vectorizer tidak pernah melihat test
pipe_split.fit(train["text"], train["label"])

pred_test = pipe_split.predict(test["text"])
print(classification_report(test["label"], pred_test, zero_division=0))

              precision    recall  f1-score   support

     negatif       0.69      0.66      0.67       400
     positif       0.67      0.71      0.69       400

    accuracy                           0.68       800
   macro avg       0.68      0.68      0.68       800
weighted avg       0.68      0.68      0.68       800



### Eksperimen yang benar: validation set dari train

Dua pilihan, dua-duanya tanpa menyentuh `test.csv`.

In [13]:
# Pilihan A -- potong validation set dari train (cepat, cocok kalau data besar)
tr2, val = train_test_split(train, test_size=0.2, random_state=42, stratify=train["label"])
print("train:", len(tr2), "| val:", len(val), "| test (disimpan):", len(test))

for ng in [(1, 1), (1, 2)]:
    p = Pipeline([("tfidf", TfidfVectorizer(ngram_range=ng, min_df=2)),
                  ("clf", LogisticRegression(max_iter=1000))]).fit(tr2["text"], tr2["label"])
    print(f"  ngram={ng} -> f1_macro(val) =",
          round(f1_score(val["label"], p.predict(val["text"]), average="macro"), 3))

# Pilihan B -- cross-validation di dalam train (lebih stabil, cocok kalau data kecil)
cv = cross_val_score(pipe_split, train["text"], train["label"],
                     cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="f1_macro")
print("\nCV di train: mean =", round(cv.mean(), 3), "| std =", round(cv.std(), 3))

train: 1920 | val: 480 | test (disimpan): 800
  ngram=(1, 1) -> f1_macro(val) = 0.706


  ngram=(1, 2) -> f1_macro(val) = 0.725



CV di train: mean = 0.694 | std = 0.019


### Varian: `test.csv` tanpa kolom label

Kalau dosen menahan label test (gaya kompetisi), yang bisa dilakukan hanyalah memprediksi lalu
menyimpan hasilnya. Validasi tetap memakai train.

In [14]:
test_nolabel = pd.read_csv(SPLIT_DIR / "test_unlabeled.csv")
print("kolom:", list(test_nolabel.columns), "| baris:", len(test_nolabel))

pred = pipe_split.predict(test_nolabel["text"].astype(str))
submission = pd.DataFrame({"id": test_nolabel["id"], "label": pred})
submission.to_csv("submission.csv", index=False)
print(submission.head(3).to_string(index=False))
print("-> tersimpan sebagai submission.csv")

kolom: ['text', 'id'] | baris: 800
 id   label
  0 negatif
  1 positif
  2 positif
-> tersimpan sebagai submission.csv


### Varian lain yang mungkin diberikan

```python
# 3 file: train / validation / test
train = load_text_dataset(SPLIT_DIR / "train.csv")
val   = load_text_dataset(SPLIT_DIR / "valid.csv")
test  = load_text_dataset(SPLIT_DIR / "test.csv")
# pakai val untuk memilih model, test hanya untuk laporan akhir

# Satu file dengan kolom penanda split
df = load_text_dataset("data.csv")          # ada kolom 'split' berisi train/test
train, test = df[df.split == "train"], df[df.split == "test"]

# Dua folder: data/train/<kelas>/*.txt dan data/test/<kelas>/*.txt
from sklearn.datasets import load_files
btr, bte = load_files("data/train", encoding="utf-8"), load_files("data/test", encoding="utf-8")

# Format .jsonl per baris {"text": ..., "label": ...}
train = pd.read_json("train.jsonl", lines=True)
```

> **Ringkas:** kalau train dan test sudah terpisah, hapus `train_test_split` dari pipeline utama,
> `fit` di train, `transform`+`predict` di test, dan pakai validation/CV **dari train** untuk
> semua percobaan.

---
## §3 · Preprocessing  ·  `OPSIONAL — semuanya bisa dilewati`

> **Kenapa seluruh bagian ini opsional?** `TfidfVectorizer` sudah melakukan lowercase dan
> tokenisasi sendiri. Jadi pipeline tanpa satu pun langkah di §3 tetap jalan dan tetap memberi
> skor yang wajar. Preprocessing itu **penyetelan**, bukan syarat. Yang biasanya paling berdampak:
> masking URL/angka (§3a), stopword yang menjaga negasi (§3a), dan stemming untuk bahasa
> Indonesia (§3b). Sisanya sering tidak mengubah skor secara berarti — tapi **argumentasinya**
> yang diminta di laporan, jadi tetap pahami alasannya.

### Teori (slide 7–13)

| Langkah | Definisi | Contoh |
|---|---|---|
| **Sentence splitter** | tentukan apakah `.` `?` `!` benar-benar *end of sentence* | "Dr. Ayu hadir." → 1 kalimat, bukan 2 |
| **Tokenization** | kalimat → daftar token; EN/ID cukup delimiter spasi | "not enak" → `["not", "enak"]` |
| **Word segmentation** | untuk bahasa tanpa spasi (Mandarin) / compound (Jerman) | `Thecatinthehat` → `the cat in the hat` |
| **Morphological analyzer** | pisah kata jadi akar + afiks | writing → writ + ing |
| **Lemmatization** | kata → **lemma** (kata dasar valid, butuh POS) | writing → **write** |
| **Stemming** | kata → potongan akar (heuristik, boleh tak valid) | writing → **writ** (Porter) |
| **Lowercase** | samakan huruf besar/kecil | FREE → free |
| **Entity masking** | ganti pola jadi token generik | `bit.ly/xx` → `urltoken`, `09066364349` → `phonetoken` |
| **Spelling correction / normalisasi** | perbaiki typo & bentuk informal / akronim | "gk" → "tidak", "yg" → "yang" |
| **Stopword elimination** | buang kata fungsi; bisa via list, bobot kata, atau filter POS (N/V/Adj) | "The place is nice but the food is not recommended" → `place nice food recommended` |

**Tiga jebakan yang sering ditanya asisten:**

1. **Lemmatization vs stemming vs morphological analyzer** — tiga hal berbeda (tabel di atas).
   Stemming cepat & tanpa kamus tapi hasilnya bisa bukan kata; lemmatization butuh kamus + POS tapi hasilnya valid.
2. **Maximum matching** (algoritma segmentasi greedy, slide 10): pointer di awal string → cari kata
   **terpanjang** di kamus yang cocok mulai pointer → geser pointer ke akhir kata itu → ulangi.
3. **Stopword bisa merusak sentimen.** Contoh slide sendiri membuang "not" dari
   *"the food is **not** recommended"* → sisa `food recommended` yang berbalik makna.
   Untuk analisis sentimen, **kecualikan kata negasi** dari daftar stopword (atau pakai bigram).

### 3a · Preprocessing bahasa Inggris (NLTK, dengan fallback)

In [15]:
import re

NLTK_OK = True
try:
    import nltk

    def punya(path):
        # sebagian resource tersimpan sebagai folder, sebagian sebagai .zip -> cek dua-duanya
        for kandidat in (path, path + ".zip"):
            try:
                nltk.data.find(kandidat)
                return True
            except LookupError:
                continue
        return False

    # Unduh hanya paket yang belum ada -- kalau sudah terpasang, sel ini jalan tanpa internet.
    for pkg, path in [("punkt", "tokenizers/punkt"), ("punkt_tab", "tokenizers/punkt_tab"),
                      ("stopwords", "corpora/stopwords"), ("wordnet", "corpora/wordnet"),
                      ("omw-1.4", "corpora/omw-1.4")]:
        if not punya(path):
            try:
                nltk.download(pkg, quiet=True)
            except Exception:
                pass
    from nltk.tokenize import word_tokenize
    from nltk.corpus import stopwords
    from nltk.stem import PorterStemmer, WordNetLemmatizer
    STOP_EN = set(stopwords.words("english"))
    stemmer, lemmatizer = PorterStemmer(), WordNetLemmatizer()
    word_tokenize("test run")           # paksa error kalau data punkt belum ada
except Exception as e:
    NLTK_OK = False
    print("NLTK tidak siap (", type(e).__name__, ") -> pakai fallback regex")
    STOP_EN = {"i","me","my","we","our","you","your","he","she","it","they","them","this","that",
               "these","those","am","is","are","was","were","be","been","being","have","has","had",
               "do","does","did","a","an","the","and","but","if","or","because","as","of","at","by",
               "for","with","to","from","in","out","on","off","then","so","than","too","very","will",
               "just","now","s","t","can","don","should"}
    def word_tokenize(s):               # noqa: F811
        return re.findall(r"[a-z0-9']+", s.lower())

NEGATION = {"no", "not", "never", "nor", "n't", "cannot"}
STOP_EN_SAFE = STOP_EN - NEGATION       # pertahankan negasi untuk task sentimen
print("NLTK siap:", NLTK_OK, "| jumlah stopword:", len(STOP_EN))

NLTK siap: True | jumlah stopword: 198


In [16]:
# Placeholder ditulis sebagai satu kata alfabet (bukan "<URL>"), karena word_tokenize
# akan memecah "<URL>" menjadi ["<", "URL", ">"] dan mask-nya jadi rusak.
# Urutan penting: email dicek sebelum URL, angka panjang sebelum angka biasa.
def mask_entities(text: str) -> str:
    text = re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+", " emailtoken ", text)
    text = re.sub(r"http\S+|www\.\S+|\b\S+\.(?:com|org|net|ly|id)\S*", " urltoken ", text)
    text = re.sub(r"@\w+", " usertoken ", text)
    text = re.sub(r"\b\d{7,}\b", " phonetoken ", text)
    text = re.sub(r"[$£€]\s?\d[\d,.]*", " moneytoken ", text)
    text = re.sub(r"\b\d+\b", " numtoken ", text)
    return text


def preprocess_en(text, use="lemma", drop_stop=True, keep_negation=True, mask=True):
    # text -> list of token. use = "none" | "stem" | "lemma"
    text = text.lower()
    if mask:
        text = mask_entities(text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha()]
    if drop_stop:
        stop = STOP_EN_SAFE if keep_negation else STOP_EN
        tokens = [t for t in tokens if t not in stop]
    if use == "stem" and NLTK_OK:
        tokens = [stemmer.stem(t) for t in tokens]
    elif use == "lemma" and NLTK_OK:
        tokens = [lemmatizer.lemmatize(t, pos="v") for t in tokens]
    return tokens


demo = "URGENT!! Call 09066364349 now, claim your FREE £10,000 prize at bit.ly/win"
print("mentah   :", demo)
print("masking  :", mask_entities(demo.lower()))
print("no-stem  :", preprocess_en(demo, use="none"))
print("stemming :", preprocess_en(demo, use="stem"))
print("lemma    :", preprocess_en(demo, use="lemma"))

mentah   : URGENT!! Call 09066364349 now, claim your FREE £10,000 prize at bit.ly/win
masking  : urgent!! call  phonetoken  now, claim your free  moneytoken  prize at  urltoken 
no-stem  : ['urgent', 'call', 'phonetoken', 'claim', 'free', 'moneytoken', 'prize', 'urltoken']
stemming : ['urgent', 'call', 'phonetoken', 'claim', 'free', 'moneytoken', 'prize', 'urltoken']


lemma    : ['urgent', 'call', 'phonetoken', 'claim', 'free', 'moneytoken', 'prize', 'urltoken']


In [17]:
# Bukti perbedaan stemming vs lemmatization (materi slide 11)
if NLTK_OK:
    rows = [(w, stemmer.stem(w), lemmatizer.lemmatize(w, pos="v"))
            for w in ["writing", "studies", "running", "better", "caring", "flies", "was"]]
    print(pd.DataFrame(rows, columns=["kata", "Porter stem", "lemma (pos=v)"]).to_string(index=False))
else:
    print("Lewati -- NLTK tidak tersedia.")

   kata Porter stem lemma (pos=v)
writing       write         write
studies       studi         study
running         run           run
 better      better        better
 caring        care          care
  flies         fli           fly
    was          wa            be


### 3b · Preprocessing bahasa Indonesia

Kalau `Sastrawi` terpasang (`pip install Sastrawi`) sel ini memakainya; kalau tidak, fallback
ke stopword list manual + tanpa stemming. Bahasa Indonesia berimbuhan berat
(*mengecewakan* → *kecewa*), jadi stemming lumayan berpengaruh.

In [18]:
STOP_ID = {"yang","dan","di","ke","dari","ini","itu","untuk","dengan","pada","adalah","ada",
           "saya","kamu","dia","kami","kita","mereka","akan","sudah","telah","juga","atau",
           "karena","agar","saja","oleh","sebagai","dalam","para","lah","pun","nya","banget",
           "sekali","sangat","pokoknya","tetapi","tapi"}
NEGASI_ID = {"tidak", "bukan", "tanpa", "jangan", "belum", "kurang"}
STOP_ID_SAFE = STOP_ID - NEGASI_ID

SLANG_ID = {"gk": "tidak", "ga": "tidak", "gak": "tidak", "nggak": "tidak", "engga": "tidak",
            "yg": "yang", "dgn": "dengan", "tdk": "tidak", "sy": "saya", "bgt": "banget",
            "udh": "sudah", "blm": "belum", "trs": "terus", "krn": "karena"}

try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    stemmer_id = StemmerFactory().create_stemmer()
    SASTRAWI_OK = True
except Exception:
    stemmer_id, SASTRAWI_OK = None, False


def preprocess_id(text, do_stem=True, drop_stop=True):
    text = text.lower()
    text = mask_entities(text)
    tokens = re.findall(r"[a-z]+", text)
    tokens = [SLANG_ID.get(t, t) for t in tokens]
    if drop_stop:
        tokens = [t for t in tokens if t not in STOP_ID_SAFE]
    if do_stem and SASTRAWI_OK:
        tokens = [stemmer_id.stem(t) for t in tokens]
    return tokens


print("Sastrawi tersedia:", SASTRAWI_OK)
print(preprocess_id("Barang datang rusak dan pelayanannya lambat sekali, gk recommended"))

Sastrawi tersedia: False
['barang', 'datang', 'rusak', 'pelayanannya', 'lambat', 'tidak', 'recommended']


### 3c · Menyambungkan preprocessing ke vectorizer

Ada dua cara. **Cara B dipakai sebagai alur utama** di notebook ini; Cara A ditulis sebagai
komentar — tinggal di-uncomment kalau kamu lebih suka cara itu. Jangan pakai keduanya sekaligus.

In [19]:
from sklearn.feature_extraction.text import CountVectorizer

docs = df_en["text"].tolist()

# =============================================================
# ALUR YANG DIPAKAI -- Cara B: bersihkan dulu jadi string biasa,
# lalu vectorizer memakai setelan default.
# =============================================================
docs_clean = [" ".join(preprocess_en(d, use="lemma")) for d in docs]

print("mentah :", docs[0][:72])
print("bersih :", docs_clean[0][:72])

# =============================================================
# ALTERNATIF -- Cara A: vectorizer memanggil tokenizer kustom kita.
# Uncomment blok ini kalau mau memakainya, dan JANGAN pakai bersamaan dengan Cara B.
# -------------------------------------------------------------
# vec_a = CountVectorizer(tokenizer=lambda s: preprocess_en(s, use="lemma"),
#                         lowercase=False,      # lowercase sudah ditangani preprocess_en
#                         token_pattern=None)   # wajib, kalau tidak sklearn memberi warning
# Xa = vec_a.fit_transform(docs)
# print("Cara A ->", Xa.shape)

mentah : My train is delayed again, I should arrive by six if nothing else happen
bersih : train delay arrive six nothing else happen


> **Kenapa Cara B yang dipilih:** hasil bersihnya berupa string yang bisa diperiksa mata
> (`print(docs_clean[0])`), jadi kalau skornya aneh kamu bisa langsung melihat apa yang salah di
> preprocessing. Cara A lebih ringkas tapi hasil antaranya tersembunyi di dalam vectorizer.
>
> Kalau memakai Cara A, `token_pattern=None` wajib diisi agar sklearn tidak memunculkan warning,
> dan `lowercase=False` supaya lowercase tidak dikerjakan dua kali.

### 3d · Sisa langkah preprocessing di slide  ·  `SEMUA OPSIONAL`

Bagian 3a–3c sudah mencakup lowercase, tokenization, stopword, stemming, lemmatization, masking,
dan normalisasi slang. Empat langkah berikut ada di slide tapi jarang dipakai untuk klasifikasi
teks pendek — tetap ditulis di sini supaya kalau ditanya, kodenya ada.

**Sentence splitter** (slide 8) — memutus teks jadi kalimat. Sulitnya: titik tidak selalu akhir
kalimat (`Dr.`, `No.`, `3.14`). Berguna kalau dokumennya panjang dan kamu mau mengklasifikasi
per kalimat, atau untuk analisis sentimen per aspek.

In [20]:
SINGKATAN = {"dr", "prof", "mr", "mrs", "ms", "no", "vs", "st", "jl", "hal", "dll", "dsb"}


def split_kalimat(teks):
    if NLTK_OK:
        from nltk.tokenize import sent_tokenize
        try:
            return sent_tokenize(teks)
        except Exception:
            pass
    # fallback: pisah setelah . ! ? yang diikuti spasi + huruf kapital, kecuali setelah singkatan
    potongan, buffer = [], ""
    for bagian in re.split(r"(?<=[.!?])\s+", teks):
        buffer = (buffer + " " + bagian).strip()
        kata_akhir = re.sub(r"[^a-z]", "", buffer.split()[-1].lower()) if buffer.split() else ""
        if kata_akhir not in SINGKATAN:
            potongan.append(buffer)
            buffer = ""
    if buffer:
        potongan.append(buffer)
    return potongan


contoh = "Dr. Ayu mengajar NLP. Kelasnya jam 8 pagi! Apakah kamu ikut? Harga bukunya Rp3.500."
for i, k in enumerate(split_kalimat(contoh), 1):
    print(i, "|", k)

1 | Dr. Ayu mengajar NLP.
2 | Kelasnya jam 8 pagi!
3 | Apakah kamu ikut?
4 | Harga bukunya Rp3.500.


**Maximum matching** (slide 10) — algoritma *greedy* baseline untuk *word segmentation*, dipakai
pada bahasa tanpa spasi (Mandarin) atau kata majemuk Jerman. Algoritmanya persis seperti di slide:
pointer di awal string → cari kata **terpanjang** di kamus yang cocok mulai dari pointer → geser
pointer ke akhir kata itu → ulangi.

In [21]:
def maximum_matching(teks, kamus, maks_kata=None):
    maks_kata = maks_kata or max(len(w) for w in kamus)
    hasil, i = [], 0
    while i < len(teks):
        for panjang in range(min(maks_kata, len(teks) - i), 0, -1):   # coba yang TERPANJANG dulu
            kandidat = teks[i:i + panjang]
            if kandidat in kamus:
                hasil.append(kandidat)
                i += panjang
                break
        else:                                                          # tidak ada yang cocok
            hasil.append(teks[i])                                      # ambil 1 karakter, lanjut
            i += 1
    return hasil


kamus = {"the", "cat", "in", "hat", "a", "table", "he", "at", "there", "sand", "island", "is",
         "land", "and"}
print("thecatinthehat ->", maximum_matching("thecatinthehat", kamus))
print("theresandisland ->", maximum_matching("theresandisland", kamus))
print("   ^ contoh kelemahan greedy: 'there'+'sand' menang atas 'the'+'res'+'and'")

thecatinthehat -> ['the', 'cat', 'in', 'the', 'hat']
theresandisland -> ['there', 'sand', 'island']
   ^ contoh kelemahan greedy: 'there'+'sand' menang atas 'the'+'res'+'and'


**Morphological analyzer** (slide 11) — memisahkan kata jadi akar + afiks, berbeda dari stemming
(yang hanya memotong) dan lemmatization (yang mengembalikan kata dasar valid). Yang di bawah ini
versi mainan berbasis daftar afiks, cukup untuk menunjukkan konsepnya.

In [22]:
SUFIKS_EN = ["ing", "ed", "es", "s", "ly", "ness", "ment", "tion", "able", "er", "est"]
PREFIKS_EN = ["un", "re", "dis", "pre", "mis", "non"]
PREFIKS_ID = ["meng", "meny", "mem", "men", "me", "peng", "pem", "pen", "per", "ber", "ter", "di", "ke"]
SUFIKS_ID = ["kan", "an", "i", "nya"]


def analisis_morfologi(kata, prefiks, sufiks):
    akar, pre, suf = kata.lower(), [], []
    berubah = True
    while berubah:
        berubah = False
        for p in prefiks:
            if akar.startswith(p) and len(akar) - len(p) >= 3:
                pre.append(p); akar = akar[len(p):]; berubah = True; break
        for s in sufiks:
            if akar.endswith(s) and len(akar) - len(s) >= 3:
                suf.insert(0, s); akar = akar[:-len(s)]; berubah = True; break
    return pre, akar, suf


for kata in ["writing", "unhappiness", "disagreement", "rewritten"]:
    pre, akar, suf = analisis_morfologi(kata, PREFIKS_EN, SUFIKS_EN)
    print(f"{kata:15s} -> prefiks={pre} akar='{akar}' sufiks={suf}")
print()
for kata in ["mengecewakan", "pelayanannya", "berkualitas", "diterima"]:
    pre, akar, suf = analisis_morfologi(kata, PREFIKS_ID, SUFIKS_ID)
    print(f"{kata:15s} -> prefiks={pre} akar='{akar}' sufiks={suf}")

writing         -> prefiks=[] akar='writ' sufiks=['ing']
unhappiness     -> prefiks=['un'] akar='happin' sufiks=['es', 's']
disagreement    -> prefiks=['dis'] akar='agree' sufiks=['ment']
rewritten       -> prefiks=['re'] akar='written' sufiks=[]

mengecewakan    -> prefiks=['meng'] akar='ecewa' sufiks=['kan']
pelayanannya    -> prefiks=[] akar='pelay' sufiks=['an', 'an', 'nya']
berkualitas     -> prefiks=['ber'] akar='kualitas' sufiks=[]
diterima        -> prefiks=['di', 'ter'] akar='ima' sufiks=[]


> Perhatikan hasilnya tidak selalu benar (`mengecewakan` → akar `ecewa`, bukan `kecewa`) — memang
> begitulah keterbatasan pendekatan berbasis aturan tanpa kamus. Untuk kerja nyata pakai Sastrawi
> (ID) atau spaCy/NLTK (EN). Ini menjelaskan kenapa slide memisahkan *analyzer*, *stemmer*, dan
> *lemmatizer* sebagai tiga hal berbeda.

**Stopword lewat POS tag** (slide 12) — alih-alih daftar kata, saring berdasarkan kelas kata:
untuk *information retrieval*, slide menyarankan menyisakan **Noun, Verb, Adjective**.

In [23]:
def saring_pos(teks, kelas_dipakai=("N", "V", "J")):
    # NN* = noun, VB* = verb, JJ* = adjective
    if not NLTK_OK:
        print("(NLTK tidak tersedia -- langkah ini dilewati)")
        return teks.lower().split()
    import nltk
    # NLTK >= 3.9 memakai nama resource '..._eng'; versi lama memakai nama tanpa akhiran.
    ada = False
    for path in ["taggers/averaged_perceptron_tagger_eng", "taggers/averaged_perceptron_tagger"]:
        try:
            nltk.data.find(path)
            ada = True
            break
        except LookupError:
            continue
    if not ada:
        for pkg in ["averaged_perceptron_tagger_eng", "averaged_perceptron_tagger"]:
            try:
                nltk.download(pkg, quiet=True)
            except Exception:
                pass
    try:
        tagged = nltk.pos_tag(word_tokenize(teks.lower()))
    except Exception as e:
        print("(POS tagger tidak siap:", type(e).__name__, "-- langkah dilewati)")
        return teks.lower().split()
    print("  tag:", tagged)
    return [w for w, t in tagged if t[0] in kelas_dipakai]


kalimat = "The place is nice but the food is not recommended"
print("asli   :", kalimat)
print("N/V/Adj:", saring_pos(kalimat))
# Slide 13 menghasilkan "place nice food recommended" -- yaitu Noun+Adj saja, kata kerja bantu
# (is/are/was) ikut dibuang. Setara dengan membatasi kelasnya:
print("N/Adj  :", saring_pos(kalimat, kelas_dipakai=("N", "J")))

asli   : The place is nice but the food is not recommended
  tag: [('the', 'DT'), ('place', 'NN'), ('is', 'VBZ'), ('nice', 'JJ'), ('but', 'CC'), ('the', 'DT'), ('food', 'NN'), ('is', 'VBZ'), ('not', 'RB'), ('recommended', 'JJ')]
N/V/Adj: ['place', 'is', 'nice', 'food', 'is', 'recommended']
  tag: [('the', 'DT'), ('place', 'NN'), ('is', 'VBZ'), ('nice', 'JJ'), ('but', 'CC'), ('the', 'DT'), ('food', 'NN'), ('is', 'VBZ'), ('not', 'RB'), ('recommended', 'JJ')]
N/Adj  : ['place', 'nice', 'food', 'recommended']


> Bandingkan dengan contoh di slide 13 yang menghasilkan `place nice food recommended`.
> Perhatikan juga bahwa cara ini **membuang "not"** — lagi-lagi berbahaya untuk analisis sentimen.

**Spelling correction / normalisasi** (slide 7) — mengoreksi typo ke kata terdekat di kosakata.
Versi di bawah memakai `difflib` dari pustaka standar, jadi tidak butuh instalasi apa pun.

In [24]:
import difflib


def buat_kosakata(daftar_teks, min_freq=2):
    from collections import Counter
    c = Counter(w for t in daftar_teks for w in re.findall(r"[a-z]+", t.lower()))
    return {w for w, n in c.items() if n >= min_freq}


def koreksi_ejaan(tokens, kosakata, cutoff=0.85):
    hasil = []
    for t in tokens:
        if t in kosakata or len(t) <= 3:
            hasil.append(t)
        else:
            dekat = difflib.get_close_matches(t, kosakata, n=1, cutoff=cutoff)
            hasil.append(dekat[0] if dekat else t)
    return hasil


kosakata = buat_kosakata(df_en["text"], min_freq=2) | {"meeting", "tomorrow", "assignment"}
salah_ketik = ["meetng", "tommorow", "assignmnt", "prize", "zzxqy"]
print("sebelum:", salah_ketik)
print("sesudah:", koreksi_ejaan(salah_ketik, kosakata))

sebelum: ['meetng', 'tommorow', 'assignmnt', 'prize', 'zzxqy']
sesudah: ['meeting', 'tomorrow', 'assignment', 'prize', 'zzxqy']


> Hati-hati: koreksi ejaan bisa **merusak** data (nama orang, istilah teknis, slang yang justru
> penanda kelas). Pada teks media sosial, sering lebih baik menormalkan slang lewat kamus
> (seperti `SLANG_ID` di 3b) daripada mengoreksi ejaan secara otomatis.

### Peta kelengkapan: setiap langkah preprocessing di slide vs kode di notebook ini

| Langkah di slide 02a | Ada di | Status | Kalau dilewati |
|---|---|---|---|
| Sentence splitter (hal. 8) | `split_kalimat` (§3d) | **opsional** | tidak apa-apa; hanya perlu untuk dokumen panjang / analisis per kalimat |
| Tokenizer (hal. 9) | `preprocess_en`, `preprocess_id` (§3a–3b) | **wajib** | vectorizer tetap menokenisasi sendiri, jadi tetap jalan |
| Word segmentation / maximum matching (hal. 10) | `maximum_matching` (§3d) | **opsional** | hanya relevan untuk bahasa tanpa spasi |
| Morphological analyzer (hal. 11) | `analisis_morfologi` (§3d) | **opsional** | jarang dipakai langsung untuk klasifikasi |
| Lemmatization (hal. 11) | `preprocess_en(use="lemma")` | **opsional** | fitur jadi lebih banyak, skor biasanya beda tipis |
| Stemming (hal. 11) | `preprocess_en(use="stem")`, Sastrawi (§3b) | **opsional** | idem; untuk bahasa Indonesia dampaknya lebih terasa |
| Lowercase (hal. 7) | `TfidfVectorizer(lowercase=True)` — default | **opsional** | sudah otomatis dilakukan vectorizer |
| Entity masking (hal. 7) | `mask_entities` (§3a) | **opsional** | URL/angka jadi fitur unik yang tidak berguna |
| Spelling correction (hal. 7) | `koreksi_ejaan` (§3d) | **opsional** | biasanya memang dilewati; berisiko merusak data |
| Normalisasi akronim/slang (hal. 7) | `SLANG_ID` (§3b) | **opsional** | disarankan untuk teks media sosial berbahasa Indonesia |
| Stopword elimination — daftar kata (hal. 12) | `STOP_EN_SAFE`, `STOP_ID_SAFE` | **opsional** | TF-IDF sudah menekan bobot kata umum |
| Stopword elimination — bobot kata (hal. 12) | `min_df`/`max_df`, TF-IDF (§4–§5) | **opsional** | — |
| Stopword elimination — filter POS (hal. 12) | `saring_pos` (§3d) | **opsional** | jarang dipakai di luar information retrieval |

### 3e · Alur preprocessing yang dipakai  ·  `INI YANG DI-COPY`

Semua fungsi di 3a–3d sudah didefinisikan, tapi belum tentu semuanya dipanggil. Sel ini adalah
**satu-satunya tempat** yang menentukan preprocessing mana yang benar-benar jalan.
Ubah argumen atau uncomment baris sesuai kebutuhan; hasilnya (`docs_clean`) dipakai §4 dan §5.

| Argumen `preprocess_en` | Nilai | Efek |
|---|---|---|
| `use` | `"none"` / `"stem"` / `"lemma"` | tanpa normalisasi kata / Porter stemmer / WordNet lemma |
| `drop_stop` | `True` / `False` | buang stopword atau tidak |
| `keep_negation` | `True` / `False` | `True` = "not", "never" **tidak** dibuang (wajib untuk sentimen) |
| `mask` | `True` / `False` | URL, mention, angka, nominal → token generik |

In [25]:
# =============================================================
# ALUR PREPROCESSING -- ubah argumen di sini, bukan di tempat lain
# =============================================================
docs_clean = [" ".join(preprocess_en(t,
                                     use="lemma",         # "none" | "stem" | "lemma"
                                     drop_stop=True,      # buang stopword
                                     keep_negation=True,  # tapi pertahankan not/never
                                     mask=True))          # URL/angka -> token generik
              for t in df_en["text"]]

# -------------------------------------------------------------
# LANGKAH OPSIONAL TAMBAHAN dari 3d -- semuanya dimatikan secara default.
# Uncomment baris yang dibutuhkan saja; urutannya boleh diubah.
# -------------------------------------------------------------
# docs_clean = [" ".join(koreksi_ejaan(d.split(), kosakata)) for d in docs_clean]   # koreksi typo (lambat)
# docs_clean = [" ".join(saring_pos(d)) for d in docs_clean]                        # sisakan N/V/Adj (lambat)
# docs_clean = [" ".join(maximum_matching(d.replace(" ", ""), kamus)) for d in docs_clean]  # bahasa tanpa spasi
# kalimat = [k for d in df_en["text"] for k in split_kalimat(d)]                    # klasifikasi per kalimat

print("jumlah dokumen :", len(docs_clean))
print("sebelum        :", df_en["text"].iloc[0][:72])
print("sesudah        :", docs_clean[0][:72])
print("rata-rata token: %.1f -> %.1f" % (
    df_en["text"].str.split().str.len().mean(),
    np.mean([len(d.split()) for d in docs_clean])))

jumlah dokumen : 80
sebelum        : My train is delayed again, I should arrive by six if nothing else happen
sesudah        : train delay arrive six nothing else happen
rata-rata token: 13.7 -> 8.0


> Kalau ingin **melewati preprocessing sama sekali** (dan itu sah — lihat catatan di awal §3),
> ganti sel di atas dengan satu baris: `docs_clean = df_en["text"].tolist()`.
> `TfidfVectorizer` tetap melakukan lowercase dan tokenisasi sendiri.

---
## §4 · Feature extraction — token jadi angka  ·  `WAJIB`

### Teori (slide 15–17)

*Vectorization* = mengubah daftar token jadi angka. Model klasik memakai **Bag of Words (BoW)** /
**Vector Space Model**: satu **matriks term × document**, satu token = satu kolom = satu skor.
Urutan kata hilang — itulah kenapa n-gram dipakai untuk mengembalikan sedikit informasi urutan.

Tiga skema bobot sel matriks:

| Skema | Nilai sel | Kapan dipakai |
|---|---|---|
| **Boolean** | 1 kalau term ada, 0 kalau tidak | teks pendek (SMS, tweet), cocok dengan BernoulliNB |
| **TF** | frekuensi term dalam dokumen | dokumen panjang, cocok dengan MultinomialNB |
| **TF-IDF** | tf × idf | default paling aman; menekan kata yang muncul di mana-mana |

**Rumus IDF versi slide:** `idf = log(N / df)` — N = jumlah dokumen, df = jumlah dokumen yang memuat term.
Intuisi: term yang muncul di **banyak** dokumen (df besar) → idf kecil → bobot kecil.
Term langka tapi ada di dokumen ini → bobot besar → paling diskriminatif.

**N-gram:** unigram = 1 kata; bigram = 2 kata berurutan ("free prize"); trigram = 3.
`ngram_range=(1,2)` artinya unigram **dan** bigram. Ini penanganan langsung terhadap
kelemahan #3 spam word list (urutan kata).

### Alur yang dipakai — pilih SATU skema bobot  ·  `INI YANG DI-COPY`

Tiga skema di bawah saling menggantikan, bukan dipakai bersamaan. Yang aktif TF-IDF (paling aman
sebagai default); dua lainnya tinggal di-uncomment kalau soal praktikum memintanya.

In [26]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# =============================================================
# PILIH SATU -- uncomment yang dipakai, comment yang lain
# =============================================================
# vect = CountVectorizer(binary=True, ngram_range=(1, 2), min_df=2)     # 1) Boolean 0/1
# vect = CountVectorizer(ngram_range=(1, 2), min_df=2)                  # 2) TF (frekuensi)
vect = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True) # 3) TF-IDF  <- aktif

X = vect.fit_transform(docs_clean)          # docs_clean dari 3e
print("matriks term x document:", X.shape)
print("12 fitur pertama       :", list(vect.get_feature_names_out()[:12]))

matriks term x document: (80, 120)
12 fitur pertama       : ['account', 'address', 'apply', 'apply numtoken', 'approve', 'approve moneytoken', 'arrive', 'assignment', 'bank', 'book', 'bring', 'call']


> Di alur nyata, `fit_transform` hanya boleh dikenakan pada **data latih**; data uji cukup
> `vect.transform(...)`. Cara teraman: bungkus vectorizer dan classifier dalam `Pipeline`
> (lihat §6) supaya sklearn mengurusnya sendiri.

### Demo pemahaman — melihat wujud ketiga matriks  ·  `boleh dilewati`

Sel berikut **bukan bagian alur**. Tujuannya satu: memperlihatkan bedanya Boolean, TF, dan
TF-IDF pada tiga dokumen yang sama, supaya angka di slide 02a hal. 16 terlihat konkret.

In [27]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Ambil 3 dokumen ASLI dari data (yang terpendek, supaya matriksnya muat di layar).
# Matriks term x document itu lebar: 80 dokumen -> ratusan kolom, mustahil dibaca mata.
# Jadi yang dikecilkan adalah JUMLAH DOKUMEN yang dilihat, bukan datanya yang dikarang.
urut = df_en["text"].str.split().str.len().sort_values()
mini_idx = list(urut.index[:2]) + [df_en[df_en["label"] == "spam"].index[0]]
mini = df_en.loc[mini_idx, "text"].tolist()
for i, d in enumerate(mini, 1):
    print(f"D{i} [{df_en.loc[mini_idx[i-1], 'label']}] {d}")

bina  = CountVectorizer(binary=True)       # 1) Boolean / biner
tf    = CountVectorizer()                  # 2) Term frequency
tfidf = TfidfVectorizer()                  # 3) TF-IDF
Mb, Mt, Mi = bina.fit_transform(mini), tf.fit_transform(mini), tfidf.fit_transform(mini)

# Ketiga vectorizer memakai analyzer default yang sama -> vocabulary-nya identik.
# Untuk ditampilkan, ambil 12 term dengan document frequency tertinggi (yang muncul di
# lebih dari satu dokumen lebih menarik untuk dilihat).
istilah = bina.get_feature_names_out()
kolom = (pd.Series(Mb.toarray().sum(axis=0), index=istilah)
           .sort_values(ascending=False).head(12).index.tolist())


def show(M, vec, title):
    frame = pd.DataFrame(M.toarray(), columns=vec.get_feature_names_out(),
                         index=["D1", "D2", "D3"])
    print(f"\n== {title} == shape: {M.shape} (menampilkan {len(kolom)} dari {M.shape[1]} kolom)")
    print(frame[kolom].round(3).to_string())


show(Mb, bina, "Boolean")
show(Mt, tf,  "TF")
show(Mi, tfidf, "TF-IDF (L2-normalized)")

D1 [spam] Congratulations, your email won our international promotion. Contact claims@lucky-notify.com
D2 [ham] Thanks for helping with the presentation, it went really well
D3 [spam] Special discount tickets for the theatre this weekend, book now at cheap-tickets-uk.com

== Boolean == shape: (3, 33) (menampilkan 12 dari 33 kolom)
    com  the  for  at  cheap  congratulations  contact  discount  book  email  helping  international
D1    1    0    0   0      0                1        1         0     0      1        0              1
D2    0    1    1   0      0                0        0         0     0      0        1              0
D3    1    1    1   1      1                0        0         1     1      0        0              0

== TF == shape: (3, 33) (menampilkan 12 dari 33 kolom)
    com  the  for  at  cheap  congratulations  contact  discount  book  email  helping  international
D1    1    0    0   0      0                1        1         0     0      1        0              1

### Parameter `TfidfVectorizer` yang perlu diingat

| Parameter | Arti | Nilai umum |
|---|---|---|
| `ngram_range` | rentang n-gram | `(1,1)` unigram, `(1,2)` uni+bigram |
| `min_df` | buang term yang muncul di < n dokumen (buang typo/hapax) | `2` atau `0.001` |
| `max_df` | buang term yang muncul di > proporsi dokumen (stopword otomatis) | `0.9` |
| `max_features` | ambil n term paling sering saja | `5000`–`50000` |
| `stop_words` | `"english"` atau list sendiri | hati-hati dengan negasi |
| `sublinear_tf` | pakai `1+log(tf)` alih-alih `tf` | `True` sering menaikkan skor |
| `binary` | paksa tf jadi 0/1 | untuk teks pendek |
| `lowercase` | lowercase otomatis | `True` (default) |

### Jebakan: IDF sklearn ≠ IDF slide

In [28]:
# Hitung IDF manual pada 3 dokumen yang sama, lalu bandingkan dengan sklearn.
# Pakai analyzer milik vectorizer itu sendiri supaya tokenisasinya identik --
# kalau memakai .split() sendiri, hasilnya beda gara-gara tanda baca & huruf kapital.
analyzer = tfidf.build_analyzer()
doc_tokens = [analyzer(d) for d in mini]
N = len(mini)
idf_asli = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))

rows = []
for term in kolom:                                        # 12 term yang ditampilkan tadi
    dfreq = sum(term in toks for toks in doc_tokens)
    idf_slide   = np.log(N / dfreq)                       # log(N/df)   <- versi slide
    idf_sklearn = np.log((1 + N) / (1 + dfreq)) + 1       # smooth_idf  <- versi sklearn
    rows.append((term, dfreq, round(idf_slide, 3), round(idf_sklearn, 3),
                 round(idf_asli[term], 3)))

cmp = pd.DataFrame(rows, columns=["term", "df", "idf log(N/df)", "idf sklearn (manual)",
                                  "tfidf.idf_ (sklearn)"])
print(cmp.to_string(index=False))
print("\nkolom terakhir cocok dengan hitungan manual:",
      bool(np.allclose(cmp["idf sklearn (manual)"], cmp["tfidf.idf_ (sklearn)"])))

           term  df  idf log(N/df)  idf sklearn (manual)  tfidf.idf_ (sklearn)
            com   2          0.405                 1.288                 1.288
            the   2          0.405                 1.288                 1.288
            for   2          0.405                 1.288                 1.288
             at   1          1.099                 1.693                 1.693
          cheap   1          1.099                 1.693                 1.693
congratulations   1          1.099                 1.693                 1.693
        contact   1          1.099                 1.693                 1.693
       discount   1          1.099                 1.693                 1.693
           book   1          1.099                 1.693                 1.693
          email   1          1.099                 1.693                 1.693
        helping   1          1.099                 1.693                 1.693
  international   1          1.099                 1

Perbedaannya ada dua dan sering ditanya:

1. **`smooth_idf=True`** (default): `idf = ln((1+N)/(1+df)) + 1`. Tambahan `+1` di pembilang/penyebut
   mencegah pembagian nol; `+1` di luar log memastikan term yang muncul di **semua** dokumen tetap
   berbobot > 0 (dengan rumus slide, `log(N/N)=0` → term itu hilang total).
2. **`norm="l2"`** (default): tiap baris (dokumen) dinormalisasi jadi panjang 1, sehingga dokumen
   panjang tidak otomatis menang. Ini sebabnya angka di tabel TF-IDF tadi berupa desimal < 1.

Kalau ingin persis rumus slide: `TfidfVectorizer(smooth_idf=False, norm=None)`.

### Efek n-gram

In [29]:
for ng in [(1, 1), (1, 2), (1, 3)]:
    v = TfidfVectorizer(ngram_range=ng)
    v.fit(df_en["text"])
    print(ng, "-> jumlah fitur:", len(v.get_feature_names_out()))

v = TfidfVectorizer(ngram_range=(1, 2))
v.fit(df_en["text"])
print("\ncontoh bigram:", [f for f in v.get_feature_names_out() if " " in f][:10])

(1, 1) -> jumlah fitur: 514
(1, 2) -> jumlah fitur: 1446
(1, 3) -> jumlah fitur: 2370

contoh bigram: ['000 usd', '000 walmart', '08000938767 to', '08001234567 immediately', '08712460324 to', '09061701461 now', '09099726395 to', '10 am', '1000 cash', '10000 deposited']


---
## §5 · Reduksi & seleksi fitur (slide 17)  ·  `OPSIONAL`

Kenapa perlu: matriks BoW itu **sparse dan raksasa** (puluhan ribu kolom). Mengurangi fitur
mempercepat training, mengurangi overfitting, dan membuang noise.

**Cara yang disebut slide:**
1. **Lemmatization / stemming** — menggabungkan varian kata jadi satu kolom.
2. **Stopword elimination.**
3. **Ambil term dengan skor tertinggi:**
   - **TF-IDF** — `idf = 1/df` atau `idf = log(N/df)`.
   - **Mutual Information (MI)** — mengukur seberapa banyak informasi kehadiran term t memberi
     tentang label l. Berbeda dari TF-IDF, MI itu **supervised** (memakai label), jadi biasanya
     lebih tajam untuk klasifikasi. Di sklearn: `mutual_info_classif`. Kerabat dekatnya `chi2`.

In [30]:
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
X = vec.fit_transform(df_en["text"])
y = df_en["label"]
names = vec.get_feature_names_out()
print("sebelum seleksi:", X.shape)

# --- chi2 (cepat, butuh nilai non-negatif -- TF-IDF & count aman)
sel_chi = SelectKBest(chi2, k=15).fit(X, y)
top_chi = pd.Series(sel_chi.scores_, index=names).sort_values(ascending=False).head(10)

# --- mutual information (persis istilah di slide, lebih lambat)
mi = mutual_info_classif(X.toarray(), y, random_state=42)
top_mi = pd.Series(mi, index=names).sort_values(ascending=False).head(10)

print("\nTop-10 chi2:\n", top_chi.round(3).to_string())
print("\nTop-10 mutual information:\n", top_mi.round(4).to_string())
print("\nsetelah SelectKBest(k=15):", sel_chi.transform(X).shape)

sebelum seleksi: (80, 1446)



Top-10 chi2:
 the      3.466
now      1.725
claim    1.238
http     0.953
free     0.893
com      0.883
we       0.879
so       0.862
been     0.818
no       0.773

Top-10 mutual information:
 the              0.3924
by six           0.1795
months free      0.1751
to               0.1724
apply 18         0.1721
it works         0.1680
left register    0.1621
your             0.1560
just checking    0.1535
address          0.1532

setelah SelectKBest(k=15): (80, 15)


In [31]:
# Cara paling praktis saat praktikum: cukup atur df di vectorizer
for kw in [dict(), dict(min_df=2), dict(min_df=2, max_df=0.8), dict(max_features=50)]:
    v = TfidfVectorizer(**kw).fit(df_en["text"])
    print(kw, "->", len(v.get_feature_names_out()), "fitur")

{} -> 514 fitur
{'min_df': 2} -> 165 fitur
{'min_df': 2, 'max_df': 0.8} -> 165 fitur
{'max_features': 50} -> 50 fitur


---
## §6 · Classifier  ·  `WAJIB`

### Teori singkat

**Multinomial Naive Bayes** — probabilistik, generatif. Memilih kelas dengan

  P(c | d) ∝ P(c) · Π P(wᵢ | c)

"Naive" karena mengasumsikan **tiap kata independen** bila kelasnya diketahui (jelas tidak benar,
tapi praktiknya bagus). `P(w|c)` diestimasi dari hitungan kata di kelas c, dengan **Laplace/add-α
smoothing** (`alpha=1.0`) supaya kata yang tak pernah muncul di suatu kelas tidak membuat seluruh
hasil perkalian jadi 0. Sangat cepat, baseline wajib untuk klasifikasi teks.

**Decision Tree** — memilih fitur yang paling memisahkan kelas di tiap percabangan (Gini/entropy).
Mudah dibaca dan digambar (dipakai di HW2), tapi gampang overfit pada ribuan fitur sparse → batasi
`max_depth`.

**Logistic Regression** — diskriminatif linear, memodelkan `P(c|d)` langsung. Biasanya lebih akurat
dari NB pada TF-IDF, koefisiennya bisa dibaca sebagai bobot kata.

**Linear SVM (`LinearSVC`)** — mencari hyperplane dengan margin terbesar. Umumnya juara untuk teks
dimensi tinggi. Catatan: tidak punya `predict_proba` (pakai `CalibratedClassifierCV` kalau butuh).

| Model | Kecepatan | Perlu skala? | Probabilitas | Cocok untuk |
|---|---|---|---|---|
| MultinomialNB | sangat cepat | tidak (input harus non-negatif) | ya | baseline, data kecil |
| BernoulliNB | sangat cepat | fitur biner | ya | teks pendek/SMS |
| DecisionTree | sedang | tidak | ya | interpretasi, visualisasi pohon |
| LogisticRegression | cepat | TF-IDF sudah cukup | ya | default yang solid |
| LinearSVC | cepat | TF-IDF sudah cukup | tidak | akurasi terbaik untuk teks |

In [32]:
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

X_text, y = df_en["text"], df_en["label"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X_text, y, test_size=0.25, random_state=42, stratify=y)   # stratify -> proporsi kelas terjaga

models = {
    "MultinomialNB":  MultinomialNB(alpha=1.0),
    "BernoulliNB":    BernoulliNB(alpha=1.0),
    "DecisionTree":   DecisionTreeClassifier(max_depth=4, random_state=42),
    "LogReg":         LogisticRegression(max_iter=1000, class_weight="balanced"),
    "LinearSVC":      LinearSVC(class_weight="balanced"),
}

hasil = []
for nama, clf in models.items():
    pipe = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))), ("clf", clf)])
    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_te)
    cv = cross_val_score(pipe, X_text, y, cv=StratifiedKFold(4, shuffle=True, random_state=42),
                         scoring="f1_macro")
    hasil.append((nama, accuracy_score(y_te, pred),
                  f1_score(y_te, pred, average="macro"), cv.mean(), cv.std()))

print(pd.DataFrame(hasil, columns=["model", "acc(test)", "f1_macro(test)", "cv_f1_mean", "cv_std"])
        .round(3).to_string(index=False))

        model  acc(test)  f1_macro(test)  cv_f1_mean  cv_std
MultinomialNB       1.00           1.000       0.925   0.025
  BernoulliNB       0.85           0.847       0.760   0.066
 DecisionTree       0.70           0.697       0.744   0.099
       LogReg       1.00           1.000       0.925   0.025
    LinearSVC       1.00           1.000       0.925   0.025

> `sms_spam.csv` cuma 80 dokumen dan kosakata spam/ham-nya nyaris tidak beririsan, jadi skornya
> mudah menyentuh 1.00 — **jangan** ambil kesimpulan dari angka sekecil ini. Yang penting polanya:
> bandingkan beberapa model dengan `cross_val_score`, bukan satu split saja. Data nyata menyusul
> di §6c.

### Decision Tree yang bisa dibaca (seperti HW2)

In [33]:
vec_dt = TfidfVectorizer(max_features=20)
Xdt = vec_dt.fit_transform(X_tr)
dt = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xdt, y_tr)
print(export_text(dt, feature_names=list(vec_dt.get_feature_names_out())))

# Visual (butuh matplotlib):
# from sklearn.tree import plot_tree
# import matplotlib.pyplot as plt
# fig, ax = plt.subplots(figsize=(14, 6))
# plot_tree(dt, feature_names=vec_dt.get_feature_names_out(),
#           class_names=dt.classes_, filled=True, rounded=True, fontsize=8, ax=ax)
# plt.show()

|--- the <= 0.25
|   |--- with <= 0.71
|   |   |--- in <= 0.51
|   |   |   |--- class: spam
|   |   |--- in >  0.51
|   |   |   |--- class: ham
|   |--- with >  0.71
|   |   |--- class: ham
|--- the >  0.25
|   |--- now <= 0.29
|   |   |--- class: ham
|   |--- now >  0.29
|   |   |--- class: spam



### Bahasa Indonesia — pipeline yang sama, tinggal ganti tokenizer

In [34]:
pipe_id = Pipeline([
    ("tfidf", TfidfVectorizer(tokenizer=preprocess_id, lowercase=False,
                              token_pattern=None, ngram_range=(1, 2))),
    ("clf",   MultinomialNB()),
])
pipe_id.fit(df_id["text"], df_id["label"])
uji = ["pelayanannya lambat dan barangnya rusak", "makanannya enak dan tempatnya nyaman"]
for t, p in zip(uji, pipe_id.predict(uji)):
    print(f"{p:8s} <- {t}")

negatif  <- pelayanannya lambat dan barangnya rusak
positif  <- makanannya enak dan tempatnya nyaman


### §6b · Multi-class — kode yang sama, tanpa perubahan

Kalau kelasnya lebih dari dua, **tidak ada yang perlu diubah** di pipeline: sklearn otomatis
memakai strategi one-vs-rest. Yang berubah hanya cara membaca metrik — pakai `average="macro"`,
dan confusion matrix-nya jadi k×k.

In [35]:
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(
    df_multi["text"], df_multi["label"], test_size=0.3, random_state=42, stratify=df_multi["label"])

pipe_multi = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
                       ("clf",   LinearSVC())]).fit(Xm_tr, ym_tr)
pred_m = pipe_multi.predict(Xm_te)

print("kelas:", list(pipe_multi.classes_))
print(classification_report(ym_te, pred_m, zero_division=0))
print(pd.DataFrame(confusion_matrix(ym_te, pred_m, labels=pipe_multi.classes_),
                   index=["true_" + c for c in pipe_multi.classes_],
                   columns=["pred_" + c for c in pipe_multi.classes_]).to_string())
print("\nprediksi kalimat baru:",
      pipe_multi.predict(["harga saham ditutup menguat pada perdagangan hari ini",
                          "gelandang itu mencetak dua gol di babak kedua"]))

kelas: ['ekonomi', 'olahraga', 'politik', 'teknologi']
              precision    recall  f1-score   support

     ekonomi       0.80      0.57      0.67         7
    olahraga       0.86      1.00      0.92         6
     politik       0.70      1.00      0.82         7
   teknologi       0.80      0.57      0.67         7

    accuracy                           0.78        27
   macro avg       0.79      0.79      0.77        27
weighted avg       0.79      0.78      0.76        27

                pred_ekonomi  pred_olahraga  pred_politik  pred_teknologi
true_ekonomi               4              0             2               1
true_olahraga              0              6             0               0
true_politik               0              0             7               0
true_teknologi             1              1             1               4

prediksi kalimat baru: ['ekonomi' 'olahraga']


### §6c · Data nyata: Sentiment140 (2.000 tweet)

Sekarang dengan data betulan — tweet mentah, penuh typo, singkatan, dan sarkasme. Skornya turun
jauh dari 1.00 dan **itu normal**: batas atas akurasi manusia pada Sentiment140 sendiri sekitar
80-an persen. Mulai sel ini, `X_tr / X_te / y_tr / y_te` dipakai ulang oleh §7–§9 dan §11,
jadi bagian evaluasi seterusnya berjalan di atas data nyata.

In [36]:
X_text, y = df_s140["text"], df_s140["label"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X_text, y, test_size=0.25, random_state=42, stratify=y)
print("latih:", len(X_tr), "| uji:", len(X_te))

hasil = []
for nama, clf in models.items():
    pipe = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
                     ("clf", clf)])
    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_te)
    hasil.append((nama, accuracy_score(y_te, pred), f1_score(y_te, pred, average="macro")))

print(pd.DataFrame(hasil, columns=["model", "acc", "f1_macro"]).round(3).to_string(index=False))

latih: 1500 | uji: 500


        model   acc  f1_macro
MultinomialNB 0.684     0.683
  BernoulliNB 0.668     0.667
 DecisionTree 0.558     0.479
       LogReg 0.696     0.696
    LinearSVC 0.672     0.672


> Bandingkan dengan tabel di §6a: dataset mainan memberi 1.00, data nyata memberi sekitar 0,7.
> Kalau di praktikum skormu 0,99 pada data nyata, curigai leakage (duplikat antar-split, atau
> vectorizer di-`fit` pada seluruh data) sebelum senang duluan.

---
## §7 · Evaluasi  ·  `WAJIB`

Untuk TP = true positive, FP = false positive, FN = false negative:

| Metrik | Rumus | Baca sebagai |
|---|---|---|
| **Accuracy** | (TP+TN)/total | proporsi benar keseluruhan — **menyesatkan bila kelas timpang** |
| **Precision** | TP/(TP+FP) | dari yang diprediksi spam, berapa yang benar spam |
| **Recall** | TP/(TP+FN) | dari semua spam sebenarnya, berapa yang tertangkap |
| **F1** | 2·P·R/(P+R) | rata-rata harmonik precision & recall |

- **macro avg** = rata-rata metrik antar kelas, tiap kelas berbobot sama → pakai kalau kelas timpang.
- **weighted avg** = rata-rata dibobot jumlah sampel tiap kelas.
- Spam filtering: **precision kelas spam lebih penting** — email penting masuk folder spam (FP)
  jauh lebih merugikan daripada satu spam lolos (FN).

In [37]:
pipe = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
                 ("clf", LogisticRegression(max_iter=1000))]).fit(X_tr, y_tr)
pred = pipe.predict(X_te)

print(classification_report(y_te, pred, zero_division=0))

labels = sorted(y.unique())
cm = confusion_matrix(y_te, pred, labels=labels)
print("Confusion matrix (baris = aktual, kolom = prediksi):")
print(pd.DataFrame(cm, index=[f"true_{l}" for l in labels],
                       columns=[f"pred_{l}" for l in labels]).to_string())

# Lihat dokumen yang salah klasifikasi -- ini yang biasanya diminta dianalisis di laporan
salah = pd.DataFrame({"text": X_te, "aktual": y_te, "prediksi": pred})
salah = salah[salah.aktual != salah.prediksi]
print(f"\nSalah klasifikasi: {len(salah)} dari {len(y_te)} dokumen. 8 contoh pertama:")
for _, r in salah.head(8).iterrows():
    print(f"  [aktual={r.aktual:8s} prediksi={r.prediksi:8s}] {r.text[:80]}")

              precision    recall  f1-score   support

     negatif       0.67      0.76      0.71       250
     positif       0.72      0.62      0.67       250

    accuracy                           0.69       500
   macro avg       0.70      0.69      0.69       500
weighted avg       0.70      0.69      0.69       500

Confusion matrix (baris = aktual, kolom = prediksi):
              pred_negatif  pred_positif
true_negatif           191            59
true_positif            95           155

Salah klasifikasi: 154 dari 500 dokumen. 8 contoh pertama:
  [aktual=positif  prediksi=negatif ] @kegan5 OOOOOH i am so fucking jealous right now i wish i would be there toooooo
  [aktual=positif  prediksi=negatif ] Going to go take a shower &amp; get ready
  [aktual=negatif  prediksi=positif ] @Sunshineliron i missed your kiss  busy watching Reva &amp; wackadoodle from way
  [aktual=positif  prediksi=negatif ] Tiring day yesterday. Plane delayed big time. Rental car pickup closed. Extra ni


---
## §8 · Tuning dengan GridSearchCV  ·  `OPSIONAL`

`Pipeline` + `GridSearchCV` = cari kombinasi preprocessing **dan** hyperparameter model sekaligus.
Nama parameter memakai format `<nama_step>__<nama_parameter>`.

In [38]:
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([("tfidf", TfidfVectorizer()), ("clf", MultinomialNB())])

grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df":      [1, 2],
    "tfidf__sublinear_tf":[False, True],
    "clf__alpha":         [0.1, 0.5, 1.0],
}

gs = GridSearchCV(pipe, grid, cv=StratifiedKFold(4, shuffle=True, random_state=42),
                  scoring="f1_macro", n_jobs=-1)
gs.fit(X_text, y)
print("param terbaik:", gs.best_params_)
print("skor CV terbaik:", round(gs.best_score_, 3))

param terbaik: {'clf__alpha': 1.0, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': True}
skor CV terbaik: 0.681


---
## §9 · Interpretasi — kata apa yang menentukan?  ·  `OPSIONAL`

Sering diminta di laporan: tunjukkan fitur paling menentukan tiap kelas.
Ini juga versi *otomatis* dari "spam word list" manual di slide 4–5.

In [39]:
def fitur_teratas(pipe, n=10):
    vec = pipe.named_steps["tfidf"]
    clf = pipe.named_steps["clf"]
    names = np.array(vec.get_feature_names_out())
    if hasattr(clf, "coef_"):                       # LogReg / LinearSVC
        skor = clf.coef_[0]
        print(f"[{clf.classes_[0]}] <-- ", ", ".join(names[np.argsort(skor)[:n]]))
        print(f"[{clf.classes_[1]}] --> ", ", ".join(names[np.argsort(skor)[-n:][::-1]]))
    elif hasattr(clf, "feature_log_prob_"):         # Naive Bayes
        for i, c in enumerate(clf.classes_):
            # selisih log-prob antar kelas = kata yang khas kelas ini
            khas = clf.feature_log_prob_[i] - clf.feature_log_prob_.mean(axis=0)
            print(f"[{c}] ", ", ".join(names[np.argsort(khas)[-n:][::-1]]))


pipe_lr = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
                    ("clf", LogisticRegression(max_iter=1000))]).fit(X_text, y)
print("== LogisticRegression =="); fitur_teratas(pipe_lr)

pipe_nb = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
                    ("clf", MultinomialNB())]).fit(X_text, y)
print("\n== MultinomialNB =="); fitur_teratas(pipe_nb)

== LogisticRegression ==
[negatif] <--  not, miss, my, no, work, to, sorry, bad, sucks, sad
[positif] -->  you, thanks, good, happy, great, awesome, thank, love, new, follow

== MultinomialNB ==
[negatif]  sucks, sorry, sick, sad, miss, hate, work, at work, is not, wanna
[positif]  thank, thanks, thank you, thanks for, follow, awesome, good morning, you got, lovely, happy


---
## §10 · Error yang sering muncul & solusinya

| Pesan error / gejala | Penyebab | Solusi |
|---|---|---|
| `ValueError: empty vocabulary` | semua kata terbuang stopword/`min_df` | turunkan `min_df`, cek hasil preprocessing |
| `Negative values in data passed to MultinomialNB` | fitur ada yang negatif (mis. hasil SVD) | pakai `LogisticRegression`, atau `MinMaxScaler` |
| `X has n features, but expects m` | vectorizer di-`fit` ulang pada data uji | pakai `Pipeline`, atau `transform` saja (bukan `fit_transform`) di data uji |
| Akurasi 100% mencurigakan | data leakage / duplikat antara train & test | `drop_duplicates()` sebelum split, jangan `fit` vectorizer di seluruh data |
| Akurasi tinggi tapi recall 1 kelas ~0 | kelas timpang | `stratify=y`, `class_weight="balanced"`, lihat `f1_macro` bukan accuracy |
| `ConvergenceWarning` di LogisticRegression | iterasi kurang | `max_iter=1000` |
| `UserWarning: token_pattern will not be used` | pakai `tokenizer=` kustom | set `token_pattern=None` |
| `LookupError: Resource punkt not found` | data NLTK belum diunduh | `nltk.download("punkt"); nltk.download("punkt_tab")` |
| `LinearSVC has no predict_proba` | SVM tak menghasilkan probabilitas | `CalibratedClassifierCV(LinearSVC())` atau pakai `decision_function` |
| Training super lambat | dataset jutaan baris + `mutual_info_classif` | subsample, pakai `max_features`, ganti MI dengan `chi2` |

---
## §11 · Lampiran  ·  `OPSIONAL`

### Kumpulan rumus

```
TF-IDF (slide)     : tfidf(t,d) = tf(t,d) × log(N / df(t))
TF-IDF (sklearn)   : tfidf(t,d) = tf(t,d) × [ ln((1+N)/(1+df(t))) + 1 ],  lalu baris di-L2-normalize
Naive Bayes        : P(c|d) ∝ P(c) · Π P(wi|c)
P(w|c) + smoothing : (count(w,c) + α) / (Σ_w' count(w',c) + α·|V|)
Precision          : TP / (TP + FP)
Recall             : TP / (TP + FN)
F1                 : 2·P·R / (P + R)
Cosine similarity  : (A·B) / (||A||·||B||)
```

### Opsional: LSA / SVD (slide 02b 7–9)

Slide 02b membahas SVD sebagai *count-based distributed representation*: matriks term×document
`A = U · S · Vᵀ`, dengan U dari eigenvector `AAᵀ` (term), V dari eigenvector `AᵀA` (dokumen),
dan S diagonal berisi **singular value** = akar eigenvalue terurut menurun.
Ambil k dimensi teratas → dokumen jadi vektor **latent semantic** berdimensi kecil, sehingga
sinonim yang sering muncul bersama jadi berdekatan. Proyeksi dokumen/query baru: `d = Dᵀ U S⁻¹`.

Di sklearn semuanya jadi satu baris (`TruncatedSVD` = LSA untuk matriks sparse):

In [40]:
from sklearn.decomposition import TruncatedSVD

vec = TfidfVectorizer()
X = vec.fit_transform(df_en["text"])

svd = TruncatedSVD(n_components=5, random_state=42)
Z = svd.fit_transform(X)                 # dokumen -> 5 dimensi latent
print("sebelum:", X.shape, "-> sesudah:", Z.shape)
print("variance dijelaskan:", svd.explained_variance_ratio_.sum().round(3))
print("singular value:", svd.singular_values_.round(3))

# Dipakai sebagai langkah di dalam pipeline (data nyata dari §6c).
# Catatan: hasil SVD bisa NEGATIF -> tidak boleh disambung ke MultinomialNB.
# Jumlah komponen harus cukup besar (ratusan) untuk teks; 5 komponen jelas terlalu sedikit.
for k in [5, 50, 200]:
    pipe_lsa = Pipeline([("tfidf", TfidfVectorizer(min_df=2)),
                         ("svd",   TruncatedSVD(n_components=k, random_state=42)),
                         ("clf",   LogisticRegression(max_iter=1000))]).fit(X_tr, y_tr)
    print(f"LSA k={k:3d} + LogReg -> akurasi {accuracy_score(y_te, pipe_lsa.predict(X_te)):.3f}")

pipe_plain = Pipeline([("tfidf", TfidfVectorizer(min_df=2)),
                       ("clf", LogisticRegression(max_iter=1000))]).fit(X_tr, y_tr)
print(f"tanpa LSA (TF-IDF penuh) -> akurasi {accuracy_score(y_te, pipe_plain.predict(X_te)):.3f}")

sebelum: (80, 514) -> sesudah: (80, 5)
variance dijelaskan: 0.107
singular value: [2.054 1.584 1.342 1.318 1.285]


LSA k=  5 + LogReg -> akurasi 0.594
LSA k= 50 + LogReg -> akurasi 0.648


LSA k=200 + LogReg -> akurasi 0.706
tanpa LSA (TF-IDF penuh) -> akurasi 0.704


In [41]:
# Verifikasi manual contoh numerik di slide: A = [[1,0,1],[2,1,1]]
A = np.array([[1., 0., 1.], [2., 1., 1.]])
AtA = A.T @ A
eigval = np.linalg.eigvalsh(A @ A.T)                 # eigenvalue AA^T (2x2, sama dgn A^T A tak-nol)
print("A A^T =\n", A @ A.T)
print("eigenvalue :", np.sort(eigval)[::-1].round(4))    # slide: 7.6055 ; 0.3944
print("singular   :", np.sqrt(np.sort(eigval)[::-1]).round(4))  # slide: 2.7578 ; 0.6281
U, S, Vt = np.linalg.svd(A, full_matrices=False)
print("S dari numpy:", S.round(4))

A A^T =
 [[2. 3.]
 [3. 6.]]
eigenvalue : [7.6056 0.3944]
singular   : [2.7578 0.6281]
S dari numpy: [2.7578 0.6281]


---
## §12 · Checklist saat praktikum

0. **Cek dulu: dikasih berapa file?** Satu file → split sendiri (§2). Dua file `train`/`test` →
   **jangan** split lagi, langsung ke §2b, dan simpan `test.csv` untuk pengukuran terakhir saja.
1. **Lihat data dulu** — `df.head()`, `df["label"].value_counts()`, cek NaN & duplikat.
2. **Split sebelum apa pun** — `train_test_split(..., stratify=y, random_state=42)`.
3. **Baseline dulu** — `TfidfVectorizer()` + `MultinomialNB()` polos, catat skornya.
4. **Baru variasikan satu per satu** — lowercase → stopword → stemming/lemma → n-gram → min_df →
   ganti model. Catat skor tiap perubahan; tabel perbandingan ini biasanya yang dinilai.
5. **Laporkan `classification_report` + confusion matrix**, bukan accuracy saja.
6. **Analisis kesalahan** — tampilkan beberapa dokumen yang salah klasifikasi dan jelaskan kenapa.
7. **Argumentasikan tiap langkah preprocessing** (persis permintaan Homework #2):
   masking URL/angka karena spam penuh link & nominal; negasi dipertahankan karena membalik sentimen;
   stemming untuk menyatukan varian imbuhan; n-gram karena urutan kata mengubah label.

### Hubungan setiap langkah ke slide

| Kode | Slide |
|---|---|
| `split_kalimat` | 02a hal. 8 (sentence splitter) |
| `mask_entities`, lowercase, tokenize | 02a hal. 7–9 (preprocessing, tokenizer) |
| `maximum_matching` | 02a hal. 10 (word segmentation, algoritma greedy) |
| `analisis_morfologi` | 02a hal. 11 (morphological analyzer: akar + afiks) |
| `PorterStemmer` vs `WordNetLemmatizer` | 02a hal. 11 (stemming vs lemmatization) |
| `STOP_EN_SAFE`, `stop_words=` | 02a hal. 12–13 (stopword elimination) |
| `saring_pos` | 02a hal. 12 (filter stopword lewat POS tag: N/V/Adj) |
| `koreksi_ejaan`, `SLANG_ID` | 02a hal. 7 (spelling correction, normalisasi akronim) |
| `CountVectorizer(binary=True)` / TF / `TfidfVectorizer` | 02a hal. 15–16 (BoW, VSM, Boolean/TF/TF-IDF) |
| `min_df`, `chi2`, `mutual_info_classif` | 02a hal. 17 (reducing feature number, MI) |
| `MultinomialNB`, `DecisionTreeClassifier`, `LinearSVC` | 02a hal. 18–19 (classification) |
| `TruncatedSVD` | 02b hal. 7–9 (SVD / LSI) |